<a href="https://colab.research.google.com/github/khrahaman/Federated-Split-Learning/blob/main/Federated_Split_Learning_(FSL)_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Federated Split Learning (FSL) Framework**

A framework that combines the Distributed Learning frameworks, Federated Learning (FL) and Split Learning (SL)

**Loading the Datasets:**

We have three Neural Networks to work with. LeNet, AlexNet and ResNet.

For LeNet we are using CIFAR-10 Dataset.

For AlexNet we are using FMNIST Dataset.

And for ResNet we are using MNIST Dataset.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# For LeNet: Dataset (CIFAR-10) Loading and Preprocessing
#(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# For AlexNet: Dataset (FMNIST) Loading and Preprocessing
#(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
#x_train, y_train = x_train[:50000], y_train[:50000]

# For ResNet18: Dataset (MNIST) Loading and Preprocessing
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, y_train = x_train[:50000], y_train[:50000]

x_train, x_test = x_train.astype("float32"), x_test.astype("float32")
y_train, y_test = y_train.flatten(), y_test.flatten()


# Split Data among clients
num_clients = 5
data_per_client = len(x_train) // num_clients
print(data_per_client)
client_data = [(x_train[i * data_per_client:(i + 1) * data_per_client],
                y_train[i * data_per_client:(i + 1) * data_per_client])
               for i in range(num_clients)]

#input_shape = (32, 32, 3) # for cifar-10
input_shape = (28,28,1) # for mnist and fmnist

num_classes = 10
batch_size = 64
epochs = 1

#Dynamic Device Assignment (early layers to CPU and later layers for gpu)
def assign_device_dynamically(layer_index, total_layers):
    return '/CPU:0' if layer_index < total_layers // 2 else '/GPU:0'


**As AlexNet and ResNet are more complex Neural Network compared to LeNet, we augment the dataset to allow them to train on more data-points for better convergence:**

In [ ]:
#augmentation for alexnet and resnet

import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# For AlexNet: Dataset (FMNIST) Loading and Preprocessing
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# For ResNet18: Dataset (MNIST) Loading and Preprocessing
#(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Define data augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=5,
)

# Calculate the number of augmented samples needed
num_augmented_samples = int(0.5 * len(x_train))

# Generate augmented samples
augmented_x_train = []
augmented_y_train = []
for x, y in datagen.flow(x_train, y_train, batch_size=32):
    augmented_x_train.append(x)
    augmented_y_train.append(y)
    if len(augmented_x_train) * 32 >= num_augmented_samples:
        break

# Convert lists to arrays
augmented_x_train = np.concatenate(augmented_x_train, axis=0)[:num_augmented_samples]
augmented_y_train = np.concatenate(augmented_y_train, axis=0)[:num_augmented_samples]

# Concatenate original and augmented data
x_train = np.concatenate((x_train, augmented_x_train), axis=0)
y_train = np.concatenate((y_train, augmented_y_train), axis=0)


# Split Data among clients
num_clients = 10
data_per_client = len(x_train) // num_clients
client_data = [(x_train[i * data_per_client:(i + 1) * data_per_client],
                y_train[i * data_per_client:(i + 1) * data_per_client])
               for i in range(num_clients)]

#input_shape = (32, 32, 3) # for cifar-10
#input_shape = (28,28,1) # for mnist and fmnist

num_classes = 10
batch_size = 64
epochs = 1

**Loading the Model:**

Load the model according to the selected dataset.

*Cut Layer*: A concept of "Split Learning (SL)". In SL, clients/users/edge devices split a Neural netowork in 2 (usually) portions. The client trains the Neural Network from the initial layer to the cut layer, which is comparatively the lightweight computation part of the model training. The later layers (heavy computation) is offloaded to a edge server (generally have more computational capacity compared to edge devices).

In this work, we treat the cpu as the client, which performs the computation of the initial layers of the Neural Network. The rest of the layer works are offloaded to a gpu, acting as a edge server.

In [ ]:
#LeNet (cut layer: 2nd layer)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, AveragePooling2D, Flatten, Dense



def build_split_model(input_shape=(32, 32, 3), num_classes=10):
    if high_client == 0:
        with tf.device('/cpu:0'):
            print("Using SL-approach")
            inputs_client = Input(shape=input_shape)
            # C1: 6 filters, 5x5, tanh
            x = Conv2D(6, (5, 5), activation='tanh', padding='valid', name='C1')(inputs_client)
            # S2: Average pooling 2x2, stride 2
            client_output = AveragePooling2D(pool_size=(2, 2), name='S2')(x)
            client_model = models.Model(inputs_client, outputs= client_output)
    else:
        with tf.device('/gpu:0'):
            print("training full model locally")
            inputs_client = Input(shape=input_shape)
            # C1: 6 filters, 5x5, tanh
            x = Conv2D(6, (5, 5), activation='tanh', padding='valid', name='C1')(inputs_client)
            # S2: Average pooling 2x2, stride 2
            client_output = AveragePooling2D(pool_size=(2, 2), name='S2')(x)
            client_model = models.Model(inputs_client, outputs= client_output)


    with tf.device('/gpu:0'):

        inputs_gateway = Input(shape = client_model.output.shape[1:])
        # C3: 16 filters, 5x5, tanh
        x = Conv2D(16, (5, 5), activation='tanh', padding='valid', name='C3')(inputs_gateway)
        # S4: Average pooling 2x2, stride 2
        x = AveragePooling2D(pool_size=(2, 2), name='S4')(x)
        # Flatten
        x = Flatten(name='Flatten')(x)
        # F5: Dense 120, tanh
        x = Dense(120, activation='tanh', name='F5')(x)
        # F6: Dense 84, tanh
        x = Dense(84, activation='tanh', name='F6')(x)
        # Output: Softmax for 10 classes
        outputs = Dense(num_classes, activation='softmax', name='Output')(x)
        gateway_model = Model(inputs_gateway, outputs, name='LeNet-5_CIFAR10')


    return client_model, gateway_model

In [ ]:
#AlexNet (closer to original alexnet)


from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_split_model(input_shape=(28, 28, 1), num_classes=10):

    if high_client == 0:
        with tf.device('/cpu:0'):
            print("using SL approach")
            inputs_client = Input(shape=input_shape)
            # Conv1: 32 filters (original: 96), 5x5, stride=1
            x = Conv2D(32, (5, 5), strides=1, activation='relu', padding='same', name='Conv1')(inputs_client)
            # Conv2: 64 filters (original: 256), 3x3
            x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv2')(x)
            x = MaxPooling2D(pool_size=(2, 2), name='Pool2')(x)

            client_output = x  # Output shape will be determined dynamically
            client_model = models.Model(inputs=inputs_client, outputs=client_output)
    else:
        with tf.device('/gpu:0'):
            inputs_client = Input(shape=input_shape)
            # Conv1: 32 filters (original: 96), 5x5, stride=1
            x = Conv2D(32, (5, 5), strides=1, activation='relu', padding='same', name='Conv1')(inputs_client)
            # Conv2: 64 filters (original: 256), 3x3
            x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv2')(x)
            x = MaxPooling2D(pool_size=(2, 2), name='Pool2')(x)

            client_output = x  # Output shape will be determined dynamically
            client_model = models.Model(inputs=inputs_client, outputs=client_output)

    with tf.device('/gpu:0'):
        # Conv3-5: 64 filters (original: 384/256)
        input_gateway = Input(shape=client_model.output_shape[1:])
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv3')(input_gateway)
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv4')(x)
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv5')(x)
        x = MaxPooling2D(pool_size=(2, 2), name='Pool3')(x)
        # Flatten
        x = Flatten(name='Flatten')(x)
        # Dense layers: 512 units (original: 4096)
        x = Dense(512, activation='relu', name='FC1')(x)
        x = Dropout(0.5)(x)
        x = Dense(512, activation='relu', name='FC2')(x)
        x = Dropout(0.5)(x)
        outputs = Dense(num_classes, activation='softmax', name='Output')(x)
        gateway_model = Model(input_gateway, outputs, name='AlexNet_FashionMNIST_Light')
    return client_model, gateway_model

ResNet uses the concept of residual blocks. These are a block consited of multiple layers.

We implemented 2 version of ResNet. In one, we split the Neural Network before the first residual block (total 4 blocks). So the CPU computes till the first residual block and all the later computation is offloaded to the GPU.

In another ResNet version, we place the cut layer after the first residual block. So the client (CPU) works on more layers before offloading the rest of the training to the edge server (GPU).

In [ ]:
#ResNet-18 (cut layer: 3rd layer; before the first residual block)


def build_split_model(input_shape=(28, 28, 1), num_classes=10):
    with tf.device('/cpu:0'):
        inputs_client = layers.Input(shape=input_shape)
        x = layers.Rescaling(1./255)(inputs_client)

        # Initial conv (smaller kernel, no stride)
        x = layers.Conv2D(32, (3, 3), strides=1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        client_output = layers.ReLU()(x)
        client_model = models.Model(inputs=inputs_client, outputs=client_output)

    with tf.device('/gpu:0'):
        # Residual blocks (3 stages)
        def residual_block(x, filters, downsample=False):
            shortcut = x
            stride = 2 if downsample else 1
            x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            if downsample:
                shortcut = layers.Conv2D(filters, (1, 1), strides=2, use_bias=False)(shortcut)
                shortcut = layers.BatchNormalization()(shortcut)
            x = layers.Add()([x, shortcut])
            x = layers.ReLU()(x)
            return x

        input_gateway = layers.Input(shape=client_model.output.shape[1:])
        # Stage 1-3
        x = residual_block(input_gateway, 32)  # Stage 1
        x = residual_block(x, 32)
        x = residual_block(x, 64, downsample=True)  # Stage 2
        x = residual_block(x, 64)
        x = residual_block(x, 128, downsample=True)  # Stage 3
        x = residual_block(x, 128)

        # Final layers
        x = layers.GlobalAveragePooling2D()(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        gateway_model = models.Model(inputs=input_gateway, outputs=outputs)
    return client_model, gateway_model


In [ ]:
#ResNet-18 (cut layer: 7th layer (40%); after the first residual block)

def build_split_model(input_shape=(28, 28, 1), num_classes=10):
    with tf.device('/cpu:0'):
        inputs_client = layers.Input(shape=input_shape)
        x = layers.Rescaling(1./255)(inputs_client)

        # Initial conv (smaller kernel, no stride)
        x = layers.Conv2D(32, (3, 3), strides=1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        #First residual block (stage 1)
        shortcut = x

        x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, shortcut])
        client_output = layers.ReLU()(x)
        client_model = models.Model(inputs=inputs_client, outputs=client_output)


    with tf.device('/gpu:0'):
        # Residual blocks (3 stages)
        def residual_block(x, filters, downsample=False):
            shortcut = x
            stride = 2 if downsample else 1
            x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            if downsample:
                shortcut = layers.Conv2D(filters, (1, 1), strides=2, use_bias=False)(shortcut)
                shortcut = layers.BatchNormalization()(shortcut)
            x = layers.Add()([x, shortcut])
            x = layers.ReLU()(x)
            return x

        input_gateway = layers.Input(shape=client_model.output.shape[1:])
        # Stage 2-3
        #x = residual_block(input_gateway, 32)  # Stage 1
        #x = residual_block(x, 32)
        x = residual_block(input_gateway, 64, downsample=True)  # Stage 2
        x = residual_block(x, 64)
        x = residual_block(x, 128, downsample=True)  # Stage 3
        x = residual_block(x, 128)

        # Final layers
        x = layers.GlobalAveragePooling2D()(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        gateway_model = models.Model(inputs=input_gateway, outputs=outputs)
    return client_model, gateway_model

**Initializing the Training:**

The concept is if a client's computational capacity is limited, we use the SL approach to complete its local training in FL.

For LeNet and AlexNet code, the condition is established if a client has high computational capacity (high_client) then it performs the whole training in its own device (the CPU). if not, then it splits the work with a edge server (GPU).

ResNet is the most complex model out of the three and usually takes a great amout of time on the CPU only. So while running ResNet, it is advised to use the SL approach.

Change the "high_end_client" list to set how many clients you want to treat as clients with high computational capacity (means they will not split the Neural Network, and perform the whole training in one device). To experiment with mixture of high and low end clients (clients with high and low computational capacity), uncomment and update the "high_end_client" list. Out of the total clients, whose indices mathces the list will be treated as high end clients and trained accorningly and the rest will be trained in SL manner.

In [ ]:
import time

global_accs = []

global high_client
high_client = 0


def evaluate_global_model(cpu_model, gpu_model):
    batch_size = 512
    num_samples = len(x_test)
    preds = []
    for i in range(0, num_samples, batch_size):
        batch_x = x_test[i:i+batch_size]
        x_intermediate = cpu_model(batch_x, training=False)
        logits = gpu_model(x_intermediate, training=False)
        batch_preds = tf.argmax(logits, axis=1)
        preds.extend(batch_preds.numpy())
    preds = tf.convert_to_tensor(preds)
    acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_test), tf.float32))
    global_accs.append(acc)
    print(f"Global Test Accuracy: {acc.numpy():.4f}")


#high_end_client = [0,1,2,3,4,5,6,7,8,9]
high_end_client = []
#Training
cpu_models, gpu_models = [], []
for _ in range(num_clients):
    if _ in high_end_client:
        high_client = 1
    else:
        high_client = 0

    cpu, gpu = build_split_model()
    cpu_models.append(cpu)
    gpu_models.append(gpu)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=1000,
    decay_rate=0.9
)
optimizers = [tf.keras.optimizers.Adam(learning_rate=lr_schedule) for _ in range(num_clients)]


#FedAvg
def average_trainable_weights_weighted(model_list, sample_counts):
    new_weights = []
    total_samples = sum(sample_counts)
    for weights in zip(*[model.trainable_weights for model in model_list]):
        weighted_sum = sum(w.numpy() * (count / total_samples)
                           for w, count in zip(weights, sample_counts))
        new_weights.append(weighted_sum)
    return new_weights


#Assigns new (averaged) weights back to each client's models
def set_trainable_weights(model, new_weights):
    for var, new in zip(model.trainable_weights, new_weights):
        var.assign(new)



client_accuracies = [[] for _ in range(num_clients)]
client_time = [[] for _ in range(num_clients)]



# Training loop
for epoch in range(epochs):
    print(f"\n🌀 Epoch {epoch + 1}/{epochs}")

    #start_time = time.time()

    for client_id, (x_client, y_client) in enumerate(client_data):


        start_time = time.time()

        train_ds = tf.data.Dataset.from_tensor_slices((x_client, y_client)).shuffle(1000).batch(batch_size)
        cpu_model = cpu_models[client_id]
        gpu_model = gpu_models[client_id]

        for step, (x_batch, y_batch) in enumerate(train_ds):
            with tf.GradientTape() as tape:
                with tf.device('/CPU:0'):
                    x_intermediate = cpu_model(x_batch, training=True)
                with tf.device('/GPU:0'):
                    logits = gpu_model(x_intermediate, training=True)
                    loss = loss_fn(y_batch, logits)
                    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(logits, axis=1), tf.cast(y_batch, tf.int64)), tf.float32))



            #Applies gradients to both CPU and GPU model portions
            #maintaining joint training across split architecture
            grads = tape.gradient(loss, cpu_model.trainable_variables + gpu_model.trainable_variables)
            optimizers[client_id].apply_gradients(zip(grads, cpu_model.trainable_variables + gpu_model.trainable_variables))


            if step % 100 == 0:
                print(f"Client {client_id}, Step {step}, Loss: {loss.numpy():.4f}, Accuracy: {acc.numpy():.4f}")




            client_accuracies[client_id].append(acc.numpy())


        end_time = time.time()
        time_taken = end_time - start_time
        client_time[client_id].append(time_taken)
        print(f"Time taken: {time_taken}")



    sample_counts = [len(data[0]) for data in client_data]
    avg_cpu_weights = average_trainable_weights_weighted(cpu_models, sample_counts)
    avg_gpu_weights = average_trainable_weights_weighted(gpu_models, sample_counts)


    for cpu_model in cpu_models:
        set_trainable_weights(cpu_model, avg_cpu_weights)

    for gpu_model in gpu_models:
        set_trainable_weights(gpu_model, avg_gpu_weights)


    evaluate_global_model(cpu_models[0], gpu_models[0])
